# Payment Card Fraud Analytics Project

## Data Preparation

This notebook prepares the raw datasets for import into a normalized MySQL database that will support fraud analytics and SQL-based investigation.

### Business Case

A payment services company wants to improve fraud investigation by identifying which transaction patterns, account characteristics, and linked-account relationships are most associated with fraud and financial loss.

The analysis is designed to support business questions such as:

* Which fraud patterns generate the highest financial losses?
* Which account characteristics are associated with higher fraud risk?
* Which transaction characteristics are most commonly associated with fraudulent activity?
* Can linked customer accounts reveal coordinated fraud rings?

### Objective of This Notebook

The objectives of this notebook are to:

* Load all raw datasets.
* Explore the structure and contents of each dataset.
* Assess data quality by identifying missing values, duplicate records, and data type inconsistencies.
* Validate relationships between datasets to support database normalization.
* Prepare cleaned CSV files for import into MySQL.

Python is used here for data preparation and validation. The final business analysis will be completed in MySQL using SQL queries.


## 1. Import Libraries

In [86]:
import pandas as pd
import numpy as np
import os

## 2. Load the Datasets

The project consists of five datasets that represent different aspects of the payment fraud ecosystem.

- **transactions** – Transaction-level payment records
- **account_profiles** – Customer account information
- **fraud_patterns** – Fraud scheme classifications
- **network_edges** – Relationships between linked accounts
- **time_series_stats** – Hourly aggregated transaction statistics

A relative folder path is used so the notebook can run from the GitHub repository on any team member's computer.

Because this notebook is stored inside the `notebooks` folder, `../data` means: go one folder up to the project root, then enter the `data` folder.


In [87]:
data_folder = "../data"

transactions_df = pd.read_csv(os.path.join(data_folder, "transactions.csv"))
account_profiles_df = pd.read_csv(os.path.join(data_folder, "account_profiles.csv"))
fraud_patterns_df = pd.read_csv(os.path.join(data_folder, "fraud_patterns.csv"))
network_edges_df = pd.read_csv(os.path.join(data_folder, "network_edges.csv"))
time_series_stats_df = pd.read_csv(os.path.join(data_folder, "time_series_stats.csv"))

print(os.listdir(data_folder))

['account_profiles.csv', 'network_edges.csv', 'transactions.csv', 'fraud_patterns.csv', 'time_series_stats.csv']


## 3. Dataset Overview

This section provides an overview of each dataset, including the number of rows, columns, missing values, and duplicate records. This helps identify potential data quality issues before importing the data into MySQL.

In [88]:
datasets = {
    "Transactions": transactions_df,
    "Account Profiles": account_profiles_df,
    "Fraud Patterns": fraud_patterns_df,
    "Network Edges": network_edges_df,
    "Time Series Stats": time_series_stats_df
}

summary = []

for name, df in datasets.items():
    summary.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values": df.isnull().sum().sum(),
        "Duplicate Rows": df.duplicated().sum()
    })

summary_df = pd.DataFrame(summary)

summary_df

,Dataset,Rows,Columns,Missing Values,Duplicate Rows
0,Transactions,1000000,23,982857,0
1,Account Profiles,50000,23,0,0
2,Fraud Patterns,7,12,0,0
3,Network Edges,7411,6,3000,0
4,Time Series Stats,26280,10,0,0


### Key Observations

The initial inspection indicates that all five datasets loaded successfully and contain no duplicate records.

Although the `transactions` dataset reports a large number of missing values, these are expected because the `fraud_pattern` field is only populated for fraudulent transactions. Legitimate transactions do not have an associated fraud pattern.

Similarly, the missing values in the `network_edges` dataset occur only in the `ring_id` column. These represent account relationships that are not associated with a known fraud ring and therefore do not require data cleaning.

Overall, the datasets are of high quality and require minimal preprocessing before being imported into the relational database.

## 4. Data Cleaning



## 4.1 Transactions Dataset

The `transactions` dataset is the primary fact table of the project. Each record represents a single payment transaction and contains transaction details, merchant information, device characteristics, behavioural indicators, and fraud labels.

The objective of this inspection is to:
- Understand the dataset structure.
- Verify the data types.
- Identify missing values and duplicate records.
- Assess whether any attributes require cleaning or normalization before importing into MySQL.

In [89]:
transactions_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 23 columns):
 #   Column               Non-Null Count    Dtype  
---  ------               --------------    -----  
 0   transaction_id       1000000 non-null  object 
 1   account_id           1000000 non-null  object 
 2   timestamp            1000000 non-null  object 
 3   hour_of_day          1000000 non-null  int64  
 4   day_of_week          1000000 non-null  int64  
 5   is_weekend           1000000 non-null  int64  
 6   amount               1000000 non-null  float64
 7   merchant_category    1000000 non-null  object 
 8   mcc_code             1000000 non-null  int64  
 9   merchant_country     1000000 non-null  object 
 10  card_present         1000000 non-null  int64  
 11  device_type          1000000 non-null  object 
 12  device_known         1000000 non-null  int64  
 13  ip_risk_score        1000000 non-null  float64
 14  is_foreign_txn       1000000 non-null  int64  
 15 

In [90]:
transactions_df.head()

,transaction_id,account_id,timestamp,hour_of_day,day_of_week,is_weekend,amount,merchant_category,mcc_code,merchant_country,...,ip_risk_score,is_foreign_txn,time_since_last_s,velocity_1h,amount_vs_avg_ratio,account_age_days,has_2fa,credit_limit,is_fraud,fraud_pattern
0,TXN000000001,ACC0016173,2023-02-21 08:02:38,8,1,0,168.42,travel,4511,CA,...,53.2,1,21,3,2.6423,3256,1,3958.46,0,NaN
1,TXN000000002,ACC0011196,2024-05-12 23:13:34,23,6,1,85.78,online_retail,5999,AU,...,25.3,1,234,1,0.7279,1527,1,3553.35,0,NaN
2,TXN000000003,ACC0001181,2023-09-22 23:28:21,23,4,0,20.15,pharmacy,5912,CA,...,21.3,1,85,1,0.1851,2230,1,4362.57,0,NaN
3,TXN000000004,ACC0037105,2022-09-28 23:26:38,23,2,0,62.49,grocery,5411,US,...,13.7,0,98,0,1.5223,1863,1,3194.84,0,NaN
4,TXN000000005,ACC0028471,2023-02-23 17:54:13,17,3,0,71.68,online_retail,5999,US,...,9.7,0,721,2,0.7724,1728,0,11850.06,0,NaN


In [91]:
# Create cleaned transactions dataframe only once
transactions_clean_df = transactions_df.drop(
    columns=[
        "account_age_days",
        "credit_limit",
        "has_2fa"
    ]
).copy()

# Convert datetime columns
transactions_clean_df["timestamp"] = pd.to_datetime(
    transactions_clean_df["timestamp"]
)

time_series_stats_df["hour"] = pd.to_datetime(
    time_series_stats_df["hour"]
)

# Check data types
transactions_clean_df.info()
time_series_stats_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 20 columns):
 #   Column               Non-Null Count    Dtype         
---  ------               --------------    -----         
 0   transaction_id       1000000 non-null  object        
 1   account_id           1000000 non-null  object        
 2   timestamp            1000000 non-null  datetime64[ns]
 3   hour_of_day          1000000 non-null  int64         
 4   day_of_week          1000000 non-null  int64         
 5   is_weekend           1000000 non-null  int64         
 6   amount               1000000 non-null  float64       
 7   merchant_category    1000000 non-null  object        
 8   mcc_code             1000000 non-null  int64         
 9   merchant_country     1000000 non-null  object        
 10  card_present         1000000 non-null  int64         
 11  device_type          1000000 non-null  object        
 12  device_known         1000000 non-null  int64         
 13

### Findings

The `transactions` dataset contains individual payment transactions and serves as the central fact table in the fraud analytics database.

The dataset was inspected to verify its structure, data quality, and suitability for import into a normalized MySQL database.

The initial inspection confirmed that:

- Each transaction is uniquely identified by `transaction_id`, making it an appropriate primary key.
- No duplicate transaction records were identified.
- The `account_id` column enables a relationship with the `account_profiles` table.
- The `fraud_pattern` column can be linked to the `fraud_patterns` reference table after missing legitimate-transaction values are handled.
- The `timestamp` column in `transactions_clean_df` and the `hour` column in `time_series_stats_df` were converted from `object` to `datetime` so date and time fields are stored consistently before SQL import.
- Account-level fields such as `account_age_days`, `credit_limit`, and `has_2fa` are removed from the cleaned transaction table because these attributes belong in the `account_profiles` table.

Overall, the transactions dataset is complete, internally consistent, and ready for import into the normalized MySQL database.

## 4.2 Account Profiles Dataset

The `account_profiles` dataset contains customer account information and account-level risk characteristics. Unlike the transactions dataset, each record represents a single customer account rather than an individual transaction.

This inspection aims to:
- Verify data completeness.
- Confirm the uniqueness of the account identifier.
- Assess whether account-level attributes require cleaning.
- Determine whether these attributes should remain in this table or be normalized into related tables.

In [92]:
account_profiles_df.describe(include="all")

,account_id,account_age_days,credit_limit,home_country,risk_score,is_high_risk,avg_txn_amount,avg_monthly_txns,has_2fa,account_type,...,max_amount,fraud_count,fraud_amount,pct_foreign,avg_velocity,unique_countries,unique_categories,avg_ip_risk,fraud_rate,is_fraudster
count,50000,50000.00000,50000.000000,50000,50000.000000,50000.00000,50000.000000,50000.000000,50000.000000,50000,...,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
unique,50000,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,ACC0000001,NaN,NaN,US,NaN,NaN,NaN,NaN,NaN,personal,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,50000,NaN,NaN,NaN,NaN,NaN,35061,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,1837.46610,6724.527119,NaN,19.912278,0.00036,79.767290,25.628144,0.648660,NaN,...,954.421862,0.342860,250.334682,0.293094,1.048726,4.892200,8.257020,21.556126,0.017127,0.266720
std,NaN,1045.95087,6167.105846,NaN,12.005753,0.01897,51.861165,20.152675,0.477394,NaN,...,861.970568,0.648564,865.803449,0.140398,0.339632,2.334757,2.992908,5.067345,0.040386,0.442249
min,NaN,30.00000,500.000000,NaN,1.000000,0.00000,6.250000,2.000000,0.000000,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,NaN,930.00000,2861.217500,NaN,10.700000,0.00000,44.700000,12.600000,0.000000,NaN,...,448.000000,0.000000,0.000000,0.210000,0.860000,3.000000,6.000000,18.740000,0.000000,0.000000
50%,NaN,1841.50000,4873.655000,NaN,17.900000,0.00000,66.700000,20.100000,1.000000,NaN,...,750.650000,0.000000,0.000000,0.290000,1.000000,5.000000,8.000000,21.420000,0.000000,0.000000
75%,NaN,2741.00000,8387.520000,NaN,27.100000,0.00000,99.842500,32.200000,1.000000,NaN,...,1192.825000,1.000000,43.090000,0.370000,1.220000,6.000000,11.000000,24.170000,0.019231,1.000000


### Findings

The `account_profiles` dataset contains one record per customer account and will serve as a dimension table in the relational database.

The dataset was assessed for missing values, duplicate records, unique identifiers, and summary statistics.

- No missing values were identified.
- No duplicate records were detected.
- `account_id` is unique for every record and is suitable as the Primary Key.
- Numeric variables fall within expected ranges.
- No additional data cleaning was required.

## 4.3 Fraud Patterns Dataset

The `fraud_patterns` dataset serves as a reference table describing the different fraud schemes identified within the transaction data. Each fraud pattern contains a description together with aggregated statistics that summarize its characteristics.

The objective of this inspection is to:
- Verify the uniqueness of each fraud pattern.
- Check for missing or inconsistent values.
- Confirm its suitability as a lookup table in the relational database.

In [93]:
fraud_patterns_df

,fraud_pattern,description,transaction_count,fraud_share_pct,avg_amount,median_amount,pct_night_0_5,pct_foreign,pct_card_not_present,avg_velocity_1h,avg_ip_risk,pct_no_2fa
0,card_not_present,Online/CNP fraud — stolen card details used wi...,5982,34.89,771.78,268.63,41.32,58.06,73.12,3.98,55.26,42.24
1,account_takeover,Fraudster gains access to legitimate account v...,3432,20.02,1203.13,436.78,40.88,59.64,74.24,3.96,55.89,41.72
2,card_present_stolen,Physical card stolen and used at POS terminals,3120,18.20,556.84,200.27,40.83,57.24,73.17,3.95,54.99,40.61
3,friendly_fraud,Legitimate cardholder disputes valid transacti...,1726,10.07,352.61,120.80,38.99,55.97,74.68,3.93,54.91,42.47
4,atm_fraud,ATM skimming or card trapping to clone card,1216,7.09,466.26,164.53,40.87,60.44,76.23,4.04,55.60,39.39
5,money_laundering,Series of transactions to disguise illicit fun...,1011,5.90,93.59,35.42,40.36,60.63,72.21,3.97,55.79,44.02
6,identity_theft,New account opened using stolen personal infor...,656,3.83,1163.50,387.26,38.72,57.62,75.30,3.93,55.14,42.23


In [94]:
fraud_patterns_df["description"] = (
    fraud_patterns_df["description"]
    .astype(str)
    .str.replace("–", "-", regex=False)
    .str.replace("—", "-", regex=False)
    .str.replace("“", '"', regex=False)
    .str.replace("”", '"', regex=False)
    .str.replace("’", "'", regex=False)
)

### Findings

The `fraud_patterns` dataset serves as a reference table describing the different fraud schemes identified within the transaction data.

The dataset was inspected for missing values, duplicate records and unique identifiers.

- No missing values were identified.
- No duplicate records were detected.
- The `fraud_pattern` field contains seven unique fraud categories and will be used as the Primary Key.
- No additional data cleaning was required.

## 4.4 Network Edges Dataset

The `network_edges` dataset captures relationships between customer accounts based on shared identifiers such as phone numbers, email domains, IP addresses, and device IDs. This information supports fraud network analysis by identifying potentially connected accounts.

The objective of this inspection is to:
- Examine relationship types.
- Validate fraud ring identifiers.
- Assess missing values.
- Verify that the dataset can be represented as a relationship table within the database.

In [95]:
network_edges_df["shared_type"].value_counts()

shared_type
ip_address      2134
email_domain    2083
device_id       2083
phone           1111
Name: count, dtype: int64

In [96]:
network_edges_df["ring_id"].value_counts(dropna=False)

ring_id
NaN         3000
RING0138      55
RING0057      55
RING0064      55
RING0069      55
            ... 
RING0162       3
RING0061       3
RING0006       3
RING0054       3
RING0021       3
Name: count, Length: 201, dtype: int64

In [97]:
network_edges_df["both_fraud"].value_counts()

both_fraud
1    4411
0    3000
Name: count, dtype: int64

In [98]:
network_edges_df.describe(include="all")

,account_a,account_b,shared_type,connection_count,ring_id,both_fraud
count,7411,7411,7411,7411.000000,4411,7411.000000
unique,3920,3929,4,NaN,200,NaN
top,ACC0012787,ACC0033150,ip_address,NaN,RING0101,NaN
freq,18,16,2134,NaN,55,NaN
mean,NaN,NaN,NaN,5.754149,NaN,0.595196
std,NaN,NaN,NaN,3.973870,NaN,0.490887
min,NaN,NaN,NaN,1.000000,NaN,0.000000
25%,NaN,NaN,NaN,3.000000,NaN,0.000000
50%,NaN,NaN,NaN,4.000000,NaN,1.000000
75%,NaN,NaN,NaN,9.000000,NaN,1.000000


### Findings

The `network_edges` dataset represents relationships between customer accounts based on shared identifiers such as phone numbers, email domains, IP addresses and device identifiers.

The dataset was assessed for missing values, duplicate records and category consistency.

- No duplicate records were identified.
- Four valid relationship types were observed (`phone`, `email_domain`, `device_id`, and `ip_address`).
- Missing values occur only in the `ring_id` column and correspond to account relationships that are not part of a known fraud ring.
- The `both_fraud` indicator contains only valid binary values (0 and 1).
- No additional data cleaning was required.

## 4.5 Time Series Statistics Dataset

The `time_series_stats` dataset contains hourly aggregated transaction metrics used for temporal fraud analysis. Unlike the transaction table, this dataset stores summarized information for each hourly period.

The objective of this inspection is to:
- Verify data completeness.
- Validate timestamp fields.
- Review summary statistics.
- Confirm that the dataset is suitable for time-based analytical queries.

In [99]:
time_series_stats_df.describe()

,hour,transaction_count,fraud_count,total_amount,avg_amount,avg_ip_risk,fraud_rate,hour_of_day,day_of_week,is_weekend
count,26280,26280.000000,26280.000000,26280.000000,26280.000000,26280.000000,26280.000000,26280.000000,26280.000000,26280.000000
mean,2023-07-02 11:30:00,38.051750,0.652321,6991.562544,183.751070,21.606374,0.017130,11.500000,3.001826,0.286758
min,2022-01-01 00:00:00,15.000000,0.000000,1661.770000,66.750000,11.480000,0.000000,0.000000,0.000000,0.000000
25%,2022-10-01 17:45:00,34.000000,0.000000,5410.165000,148.105000,19.760000,0.000000,5.750000,1.000000,0.000000
50%,2023-07-02 11:30:00,38.000000,0.000000,6684.720000,176.025000,21.530000,0.000000,11.500000,3.000000,0.000000
75%,2024-04-01 05:15:00,42.000000,1.000000,8230.637500,210.505000,23.360000,0.027778,17.250000,5.000000,1.000000
max,2024-12-30 23:00:00,67.000000,7.000000,33763.350000,985.100000,34.680000,0.218750,23.000000,6.000000,1.000000
std,NaN,6.185519,0.846763,2264.604312,52.143961,2.685101,0.022284,6.922318,2.002319,0.452256


In [100]:
time_series_stats_df["hour"].head()

0   2022-01-01 00:00:00
1   2022-01-01 01:00:00
2   2022-01-01 02:00:00
3   2022-01-01 03:00:00
4   2022-01-01 04:00:00
Name: hour, dtype: datetime64[ns]

### Findings

The `time_series_stats` dataset contains hourly aggregated transaction metrics used for temporal fraud analysis.

The dataset was assessed for missing values, duplicate records, summary statistics and data consistency.

- No missing values were identified.
- No duplicate records were detected.
- All numeric variables fall within expected ranges.
- The `hour` field represents hourly timestamps and will be stored as a `DATETIME` field in the SQL database.
- No additional data cleaning was required.


## 5. Primary Key and Foreign Key Design

Before exporting the cleaned datasets for SQL import, the relational structure of the database was defined. This step identifies the Primary Keys and Foreign Keys that will be used later when creating the MySQL tables. 

## 5.1 Primary Key Validation

### Primary Keys

| Table | Primary Key | Reason |
|---|---|---|
| `account_profiles` | `account_id` | Each row represents one unique customer account. |
| `transactions` | `transaction_id` | Each row represents one unique transaction. |
| `fraud_patterns` | `fraud_pattern` | Each row represents one unique fraud scheme. |
| `network_edges` | Composite key: `account_a`, `account_b`, `shared_type` | Each row represents a relationship between two accounts through a specific shared identifier. |
| `time_series_stats` | `hour` | Each row represents one hourly aggregated time period. |


In [101]:
print("Unique account_id in account_profiles:", account_profiles_df["account_id"].is_unique)
print("Unique transaction_id in transactions_clean:", transactions_clean_df["transaction_id"].is_unique)
print("Unique fraud_pattern in fraud_patterns:", fraud_patterns_df["fraud_pattern"].is_unique)
print("Unique hour in time_series_stats:", time_series_stats_df["hour"].is_unique)

Unique account_id in account_profiles: True
Unique transaction_id in transactions_clean: True
Unique fraud_pattern in fraud_patterns: True
Unique hour in time_series_stats: True


## 5.2 Composite Primary Key Validation 

The `network_edges` dataset does not contain a single column that uniquely identifies each relationship.

Instead, the combination of `account_a`, `account_b`, and `shared_type` uniquely identifies every network relationship. The following validation confirms that this composite key contains no duplicate records.

In [102]:
network_key_duplicates = network_edges_df.duplicated(
    subset=["account_a", "account_b", "shared_type"]
).sum()

print("Duplicate composite keys:", network_key_duplicates)

Duplicate composite keys: 0


## 5.3 Foreign Keys

After validating all primary keys, introduce the foreign keys.

| Table | Foreign Key | References | Relationship |
|---|---|---|---|
| `transactions` | `account_id` | `account_profiles(account_id)` | One account can have many transactions. |
| `transactions` | `fraud_pattern` | `fraud_patterns(fraud_pattern)` | One fraud pattern can be linked to many fraudulent transactions. |
| `network_edges` | `account_a` | `account_profiles(account_id)` | The first account in a linked-account relationship. |
| `network_edges` | `account_b` | `account_profiles(account_id)` | The second account in a linked-account relationship. |



## 5.4 Database Normalization

Database normalization was applied to improve the relational database design by reducing data redundancy and ensuring that each table stores information related to a single business entity.

During the data preparation phase, duplicate account-level attributes were identified in both the `transactions` and `account_profiles` datasets. These attributes describe characteristics of a customer account rather than individual transactions.

To validate whether these duplicated attributes could be removed from the transactions dataset, a comparison was performed using `account_id` as the common identifier.

In [103]:
validation_df = transactions_df.merge(
    account_profiles_df[
        ["account_id", "account_age_days", "credit_limit", "has_2fa"]
    ],
    on="account_id",
    suffixes=("_txn", "_profile")
)

In [104]:
validation_df[
    [
        "account_age_days_txn",
        "account_age_days_profile",
        "credit_limit_txn",
        "credit_limit_profile",
        "has_2fa_txn",
        "has_2fa_profile"
    ]
].head()

,account_age_days_txn,account_age_days_profile,credit_limit_txn,credit_limit_profile,has_2fa_txn,has_2fa_profile
0,3256,3256,3958.46,3958.46,1,1
1,1527,1527,3553.35,3553.35,1,1
2,2230,2230,4362.57,4362.57,1,1
3,1863,1863,3194.84,3194.84,1,1
4,1728,1728,11850.06,11850.06,0,0


In [105]:
print(
    "Account Age Match:",
    (validation_df["account_age_days_txn"] ==
     validation_df["account_age_days_profile"]).all()
)

print(
    "Credit Limit Match:",
    (validation_df["credit_limit_txn"] ==
     validation_df["credit_limit_profile"]).all()
)

print(
    "2FA Match:",
    (validation_df["has_2fa_txn"] ==
     validation_df["has_2fa_profile"]).all()
)

Account Age Match: True
Credit Limit Match: True
2FA Match: True


### 5.5 Proposed Entity Relationship (ER) Design

Based on the data quality assessment, primary key validation, foreign key relationships, and normalization process, the fraud analytics database will be designed using five related tables.

The proposed relational structure is:

| Table | Purpose |
|---|---|
| `account_profiles` | Stores customer account-level information and risk attributes. |
| `transactions` | Stores individual payment transaction records. |
| `fraud_patterns` | Stores known fraud scheme classifications and summary characteristics. |
| `network_edges` | Stores relationships between linked customer accounts. |
| `time_series_stats` | Stores hourly aggregated transaction statistics for temporal analysis. |

### Proposed Relationships

| Relationship | Type | Explanation |
|---|---|---|
| `account_profiles` → `transactions` | One-to-many | One account can have many transactions. |
| `fraud_patterns` → `transactions` | One-to-many | One fraud pattern can appear in many fraudulent transactions. |
| `account_profiles` → `network_edges.account_a` | One-to-many | An account can appear as the first account in many network links. |
| `account_profiles` → `network_edges.account_b` | One-to-many | An account can appear as the second account in many network links. |

The `network_edges` table references the `account_profiles` table twice through `account_a` and `account_b`. This allows the database to represent relationships between two accounts within the same account table.

The `time_series_stats` table is kept as a separate analytical summary table because it stores hourly aggregated metrics rather than individual transaction records.

### Conceptual ER Structure

```text
account_profiles
    ├── transactions
    └── network_edges
          ├── account_a → account_profiles.account_id
          └── account_b → account_profiles.account_id

fraud_patterns
    └── transactions

time_series_stats
    └── standalone hourly summary table

## 6. Fixing Foreign Key Issue: Missing Fraud Pattern for Legitimate Transactions

During the MySQL import, the transactions table may fail if legitimate transactions have a blank fraud pattern while `fraud_pattern` is defined as a foreign key.

This happens because blank or missing values cannot be matched to a value in the `fraud_patterns` reference table.

To make the relationship explicit, missing `fraud_pattern` values are replaced with `"No Fraud"`, and `"No Fraud"` is added as a valid category in the `fraud_patterns` table.


In [106]:
# Replace missing fraud patterns with "No Fraud"
transactions_clean_df["fraud_pattern"] = transactions_clean_df["fraud_pattern"].fillna("No Fraud")
transactions_clean_df["fraud_pattern"] = transactions_clean_df["fraud_pattern"].replace("", "No Fraud")

In [107]:
# Add "No Fraud" category to fraud_patterns table
new_row = {
    "fraud_pattern": "No Fraud",
    "description": "Legitimate transaction with no fraud pattern",
    "transaction_count": 0,
    "fraud_share_pct": 0,
    "avg_amount": 0,
    "median_amount": 0,
    "pct_night_0_5": 0,
    "pct_foreign": 0,
    "pct_card_not_present": 0,
    "avg_velocity_1h": 0,
    "avg_ip_risk": 0,
    "pct_no_2fa": 0
}

fraud_patterns_df = pd.concat(
    [fraud_patterns_df, pd.DataFrame([new_row])],
    ignore_index=True
)

### 7. Export Cleaned Datasets

The cleaned datasets are exported as CSV files for import into the MySQL database.

A relative output folder is used so the notebook remains GitHub-friendly. The `cleaned_data` folder will be created automatically if it does not already exist.

The datetime columns are exported using a standard `YYYY-MM-DD HH:MM:SS` format to ensure compatibility with MySQL `DATETIME` fields.

In [108]:
cleaned_folder = r"C:\Users\Winifred\Ironhack\Part time program\Projects\Project 3\Payment fraud\cleaned_data"# Define the cleaned data output folder
cleaned_folder = "../cleaned_data"

# Create the cleaned_data folder if it does not already exist
os.makedirs(cleaned_folder, exist_ok=True)

print("Cleaned data folder:", cleaned_folder)


Cleaned data folder: ../cleaned_data


In [109]:
# Dictionary of all cleaned DataFrames

datasets_to_export = {
    "transactions_clean.csv": transactions_clean_df,
    "account_profiles_clean.csv": account_profiles_df,
    "fraud_patterns_clean.csv": fraud_patterns_df,
    "network_edges_clean.csv": network_edges_df,
    "time_series_stats_clean.csv": time_series_stats_df
}

# Export all datasets

for filename, df in datasets_to_export.items():

    if filename in ["transactions_clean.csv", "time_series_stats_clean.csv"]:
        df.to_csv(
            os.path.join(cleaned_folder, filename),
            index=False,
            date_format="%Y-%m-%d %H:%M:%S"
        )
    else:
        df.to_csv(
            os.path.join(cleaned_folder, filename),
            index=False
        )

print("✅ All cleaned datasets exported successfully!")

✅ All cleaned datasets exported successfully!


In [110]:
print(transactions_clean_df.dtypes)
print(account_profiles_df.dtypes)
print(fraud_patterns_df.dtypes)
print(network_edges_df.dtypes)
print(time_series_stats_df.dtypes)

transaction_id                 object
account_id                     object
timestamp              datetime64[ns]
hour_of_day                     int64
day_of_week                     int64
is_weekend                      int64
amount                        float64
merchant_category              object
mcc_code                        int64
merchant_country               object
card_present                    int64
device_type                    object
device_known                    int64
ip_risk_score                 float64
is_foreign_txn                  int64
time_since_last_s               int64
velocity_1h                     int64
amount_vs_avg_ratio           float64
is_fraud                        int64
fraud_pattern                  object
dtype: object
account_id             object
account_age_days        int64
credit_limit          float64
home_country           object
risk_score            float64
is_high_risk            int64
avg_txn_amount        float64
avg_monthly_txns

## 8. Next Step: MySQL

After exporting the cleaned CSV files, the next step is to create the MySQL database and tables.

The SQL stage should include:

* Creating tables with appropriate data types.
* Defining primary keys and foreign keys.
* Importing the cleaned CSV files.
* Validating row counts after import.
* Answering the business questions using SQL queries.
